# Evaluación SFT (LoRA) sobre split de test

Montamos Google Drive, instalamos lo mínimo y cargamos el repositorio para usar el harness oficial.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q python-dotenv
!pip install -q -U torchao

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/desafio")
if not REPO_DIR.exists():
    !git clone -q -b dev https://github.com/gaboaurora/desafio.git {REPO_DIR}

%cd /content/desafio
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

Revisamos la GPU e importamos las librerías.

In [ ]:
import torch

if torch.cuda.is_available():
    VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", VRAM_GB)
    print("Soporta BF16:", torch.cuda.is_bf16_supported())
else:
    VRAM_GB = 0
    print("Running on CPU")

In [ ]:
import json
import pandas as pd
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from evaluation.run_baseline import (
    _close_sql_tag,
    format_table,
    load_tasks,
    load_traces,
    run_model_on_tasks,
    summarize_model,
    trace_record,
    write_summary_csv,
)
from harness.environment import SQLEnvironment

Definimos rutas y parámetros de evaluación.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MODEL_NAME = "sft_lora"
CKPT_ROOT = Path("/content/drive/MyDrive/GenAI/e2/checkpoints/sft_lora")
ADAPTER_DIR = sorted(CKPT_ROOT.glob("final_*"))[-1]

OUT_DIR = Path("/content/drive/MyDrive/GenAI/e2/eval") / MODEL_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT = "test"
K = 8
TEMPERATURE = 0.7
TOP_P = 0.95
MAX_NEW_TOKENS = 512
SEED = 2026
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

print("Adaptador:", ADAPTER_DIR)
print("Salida:   ", OUT_DIR)

Cargamos el modelo base y fusionamos el adaptador LoRA entrenado.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    device_map="auto",
)
if ADAPTER_DIR is not None:
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
    model = model.merge_and_unload()
model.eval()

Política de inferencia compatible con `SQLEnvironment` (mismo protocolo que `run_baseline.py`).

In [ ]:
def hf_policy_factory(task, sample):
    torch.manual_seed(SEED + sample)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED + sample)

    def policy(messages):
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        gen_kwargs = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=TEMPERATURE > 0,
            stop_strings=["</sql>", "</SQL>"],
            tokenizer=tokenizer,
            pad_token_id=tokenizer.pad_token_id,
        )
        if TEMPERATURE > 0:
            gen_kwargs["temperature"] = TEMPERATURE
            gen_kwargs["top_p"] = TOP_P
        with torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)
        text = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        return _close_sql_tag(text)

    return policy

Prueba rápida con 1 tarea de test antes de correr todo.

In [ ]:
tasks = load_tasks(SPLIT)
env = SQLEnvironment()
print(f"Tareas en {SPLIT}: {len(tasks)}")

ep = env.run_episode(tasks[0], hf_policy_factory(tasks[0], 0))
print("Pregunta:", tasks[0].question)
print("Expected:", tasks[0].expected_result)
print("Answer:  ", ep.final_answer, "| Reward:", ep.reward)
print(ep.completion)

Evaluación completa sobre test (guarda cada episodio en Drive y reanuda si se corta).

In [ ]:
trace_path = OUT_DIR / f"traces_{MODEL_NAME}_{SPLIT}.jsonl"
done = load_traces(trace_path)
print(f"Episodios previos encontrados: {len(done)} / {len(tasks) * K}")

with trace_path.open("a", encoding="utf-8") as f:
    def on_episode(task, sample, episode, elapsed):
        rec = trace_record(MODEL_NAME, "hf", task, sample, episode, elapsed)
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()

    episodes = run_model_on_tasks(
        MODEL_NAME,
        tasks,
        K,
        env,
        policy_factory=hf_policy_factory,
        done=done,
        on_episode=on_episode,
    )

Resumen de métricas (`pass@k`, `seen` vs `heldout`, calibración y las 3 etapas).

In [ ]:
report = {
    "config": {
        "model_id": MODEL_ID,
        "model_name": MODEL_NAME,
        "adapter_dir": str(ADAPTER_DIR),
        "split": SPLIT,
        "k": K,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "max_new_tokens": MAX_NEW_TOKENS,
        "seed": SEED,
        "n_tasks": len(tasks),
    },
    "models": {
        MODEL_NAME: summarize_model(tasks, episodes, K),
    },
}

json_path = OUT_DIR / f"{MODEL_NAME}_{SPLIT}.json"
csv_path = OUT_DIR / f"{MODEL_NAME}_{SPLIT}_summary.csv"
json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
write_summary_csv(csv_path, report)

print(format_table(report))
print(f"\nGuardado en:\n- {json_path}\n- {csv_path}\n- {trace_path}")

In [ ]:
pd.read_csv(csv_path)